<a href="https://colab.research.google.com/github/abdelrahmanahmed-663/Mini_Tasks/blob/main/Twitter_sentiment_analysis_Using_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install kaggle

Kaggle json file


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


import data set

In [ ]:
# Api of feach data from kaggle
! kaggle datasets download -d kazanova/sentiment140

Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other
100% 80.9M/80.9M [00:03<00:00, 25.2MB/s]



In [ ]:
# extact comprest dataset
from zipfile import ZipFile
dataset = '/content/sentiment140.zip'
with ZipFile (dataset  , 'r')  as zip:
  zip.extractall()
  print('The dataset is extracted')

The dataset is extracted


In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [ ]:
import nltk
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# print stopwords
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [ ]:
# data preprocessing
data = pd.read_csv('/content/training.1600000.processed.noemoticon.csv',  encoding='ISO-8859-1')
data.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [ ]:
data.describe()

,0,1467810369
count,1.599999e+06,1.599999e+06
mean,2.000001e+00,1.998818e+09
std,2.000001e+00,1.935757e+08
min,0.000000e+00,1.467811e+09
25%,0.000000e+00,1.956916e+09
50%,4.000000e+00,2.002102e+09
75%,4.000000e+00,2.177059e+09
max,4.000000e+00,2.329206e+09


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
data.shape

(1599999, 6)

In [ ]:
# naming columns
column_names = ['target', 'ids', 'date', 'flag', 'user', 'text']
data.columns = column_names


In [ ]:
# Missing values
data.isnull().sum()

,0
target,0
ids,0
date,0
flag,0
user,0
text,0


In [ ]:
#check of distripution
data['target'].value_counts()


,count
target,
4,800000
0,799999


In [ ]:
# 4 to 1
data.replace({'target':{4:1}} , inplace = True)

In [ ]:
data['target'].value_counts()

,count
target,
1,800000
0,799999


In [ ]:
port_stem = PorterStemmer()

In [ ]:
def stemming(content):
  stemmed_content = re.sub('[a-zA-Z]' ,'', content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)
  return stemmed_content

In [ ]:
data['stemmed_content'] = data['text'].apply(stemming)

In [ ]:
data.head()

,target,ids,date,flag,user,text,stemmed_content
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,' ... . !
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,@ . 50%
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....","@ , ' . ' . ? ' ."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew,@


In [ ]:
print (data['stemmed_content'])

0                  ' ... . !
1                    @ . 50%
2                           
3          @ , ' . ' . ? ' .
4                          @
                 ...        
1599994                    .
1599995    . - ! â« ://./~8
1599996                    ?
1599997               38 !!!
1599998             # @ @ @4
Name: stemmed_content, Length: 1599999, dtype: object


In [ ]:
x = data['stemmed_content'].values
y = data['target'].values

In [ ]:
print(x)

["' ... . !" '@ . 50%' '' ... '?' '38 !!!' '# @ @ @4']


In [ ]:
print(y)

[0 0 0 ... 1 1 1]


In [ ]:
x_train , x_test , y_train , y_test = train_test_split(x,y,test_size=0.2, train_size = 0.8,stratify=y,random_state=2)

In [ ]:
print (x.shape)

(1599999,)


In [ ]:
print(x_train.shape , x_test.shape)

(1279999,) (320000,)


In [ ]:
print(x_train)

['' '4000 . 2115. !' '...' ... '@ .' "@ , . . ' ."
 "@__ / &; . ' ' 3:15 &; '"]


In [ ]:
print(x_test)

['00 , ...' '/ &;' "' ' , ' - , '" ... '@41 ! 4700 !' '. .' '@ !!!! !']


In [ ]:
# Re-split x and y to ensure x_train and x_test are fresh string arrays
# This will redefine x_train, x_test, y_train, y_test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, train_size=0.8, stratify=y, random_state=2)

vectorizer = TfidfVectorizer()
x_train = vectorizer.fit_transform(x_train)
x_test = vectorizer.transform(x_test)

In [ ]:
print (x_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 199569 stored elements and shape (1279999, 9990)>
  Coords	Values
  (1, 4694)	0.6067446245268321
  (1, 3032)	0.7948968238757742
  (8, 4211)	1.0
  (24, 7125)	1.0
  (30, 589)	1.0
  (40, 105)	1.0
  (66, 4208)	1.0
  (67, 4177)	1.0
  (73, 3501)	1.0
  (74, 6549)	1.0
  (75, 813)	1.0
  (83, 3857)	1.0
  (87, 6040)	1.0
  (88, 7181)	1.0
  (96, 5020)	0.7965194182556927
  (96, 4693)	0.6046129475471168
  (97, 2819)	1.0
  (103, 7672)	1.0
  (106, 726)	1.0
  (107, 2850)	1.0
  (132, 6164)	0.6877330848977695
  (132, 6291)	0.7259636381644727
  (136, 813)	1.0
  (143, 7006)	1.0
  (153, 4349)	1.0
  :	:
  (1279853, 5396)	1.0
  (1279855, 7722)	1.0
  (1279881, 815)	1.0
  (1279883, 1364)	1.0
  (1279887, 9488)	0.4053985457471419
  (1279887, 9490)	0.9141400434868295
  (1279899, 2472)	1.0
  (1279903, 2820)	1.0
  (1279906, 2359)	1.0
  (1279908, 6357)	1.0
  (1279911, 7006)	1.0
  (1279929, 3833)	1.0
  (1279930, 3921)	1.0
  (1279931, 6582)	1.0
  (1279938, 0)

In [ ]:
print(x_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 48074 stored elements and shape (320000, 9990)>
  Coords	Values
  (0, 0)	1.0
  (6, 8047)	1.0
  (15, 2777)	1.0
  (20, 7650)	1.0
  (23, 3923)	1.0
  (27, 6114)	1.0
  (37, 2777)	1.0
  (54, 991)	1.0
  (66, 425)	0.5313525481933917
  (66, 2850)	0.45463956279154255
  (66, 3990)	0.7148197937054516
  (78, 5491)	1.0
  (89, 5397)	1.0
  (92, 7616)	1.0
  (106, 1364)	1.0
  (110, 9505)	1.0
  (114, 4493)	1.0
  (129, 5724)	1.0
  (131, 9505)	1.0
  (176, 814)	1.0
  (178, 3921)	0.6408139005972938
  (178, 5396)	0.7676962581654815
  (196, 7868)	1.0
  (198, 0)	1.0
  (204, 3502)	1.0
  :	:
  (319834, 3921)	1.0
  (319843, 3955)	1.0
  (319851, 4177)	1.0
  (319864, 3532)	1.0
  (319867, 1064)	1.0
  (319878, 3921)	0.6315326571258244
  (319878, 4692)	0.7753492780570546
  (319879, 7282)	1.0
  (319886, 8267)	1.0
  (319891, 5314)	1.0
  (319896, 3921)	1.0
  (319907, 3668)	1.0
  (319914, 7261)	1.0
  (319917, 3897)	1.0
  (319919, 4758)	1.0
  (319929, 905)	1.0
  

In [ ]:
# traing model
model = LogisticRegression(max_iter=1000)



In [ ]:
model.fit(  x_train , y_train)

LogisticRegression(max_iter=1000)

In [ ]:
x_train_predection = model.predict(x_train)
training_data_accuracy = accuracy_score(y_train , x_train_predection)

In [ ]:
print ("Accuracy score :" , training_data_accuracy)

Accuracy score : 0.5171285290066633


In [ ]:
x_test_predection = model.predict(x_test)
test_data_accuracy = accuracy_score(y_test , x_test_predection)

In [ ]:
print ("Accuracy of tesing data : ", test_data_accuracy)

Accuracy of tesing data :  0.514596875


In [ ]:
import pickle

In [ ]:
filename = 'trained_model.sav'
pickle.dump(model , open(filename, 'wb'))

In [ ]:
lodded_model = pickle.load(open('/content/trained_model.sav','rb'))

In [ ]:
x_new = x_test[200]
print (y_test[200])
prediction = lodded_model.predict(x_new)
print(prediction)
if (prediction[0] == 0):
  print ('negative tweet')
else:
    print ('positive tweet')

1
[0]
negative tweet


In [ ]:
x_new = x_test[100]
print (y_test[100])
prediction = lodded_model.predict(x_new)
print(prediction)
if (prediction[0] == 0):
  print ('negative tweet')
else:
    print ('positive tweet')

0
[0]
negative tweet
